# Train Neural Speech Decoding (Local Jupyter)

**Requirements:** GPU with CUDA support, ~16GB RAM

**Total Time:** ~16 hours (6hrs Stage 1 + 10hrs Stage 2 on A100)

## Step 0: Register Kernel (Run in Terminal First)

Your environment is already set up. Just register it as a Jupyter kernel:

```bash
conda activate /hpc/group/pearsonlab/mw582/conda_envs/neural_speech
python -m ipykernel install --user --name neural_speech --display-name "Flinker Model"
```

Then restart VS Code and select **"Flinker Model"** as your kernel before running the cells below.

## Step 1: Check GPU

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Step 2: Verify Data

Make sure HB02.h5 is in **example_data/data/**

Download from: https://data.mendeley.com/datasets/fp4bv9gtwk/2

In [ ]:
!ls -lh example_data/
print("\nShould see HB02.h5 above")

## Step 3: Update Config

In [ ]:
import json

with open('configs/AllSubjectInfo.json', 'r') as f:
    config = json.load(f)

config['Shared']['RootPath'] = './example_data/'

with open('configs/AllSubjectInfo.json', 'w') as f:
    json.dump(config, f, indent=4)

print(f"Config updated: RootPath = {config['Shared']['RootPath']}")

In [ ]:
import os

root = config['Shared']['RootPath']
print("Looking for files in:", os.path.abspath(root))
print("Files found:", os.listdir(root))

In [ ]:
!wandb login

In [ ]:
# W&B Setup - run this in terminal ONCE: wandb login
# Then this cell just verifies you're logged in
!wandb status

## Step 4: Stage 1 - Audio-to-Audio Training (a2a)

**Time:** ~6 hours on A100

In [ ]:
!python train_a2a.py \
  --OUTPUT_DIR output/a2a/HB02 \
  --trainsubject HB02 \
  --testsubject HB02 \
  --param_file configs/a2a_production.yaml \
  --batch_size 16 \
  --reshape 1 \
  --DENSITY "HB" \
  --wavebased 1 \
  --n_filter_samples 80 \
  --n_fft 256 \
  --formant_supervision 1 \
  --intensity_thres -1 \
  --epoch_num 60

In [ ]:
# Check Stage 1 completed
!ls output/a2a/HB02/*.pth | wc -l
print("Expected: 60 checkpoint files")

## Step 5: Stage 2 - ECoG-to-Audio Training (e2a)

**Time:** ~10 hours on A100

**This produces the weights you need for phoneme classification!**

In [ ]:
!python train_e2a.py \
  --OUTPUT_DIR output/e2a/resnet_HB02 \
  --trainsubject HB02 \
  --testsubject HB02 \
  --param_file configs/e2a_production.yaml \
  --batch_size 16 \
  --MAPPING_FROM_ECOG ECoGMapping_ResNet \
  --reshape 1 \
  --DENSITY "HB" \
  --wavebased 1 \
  --dynamicfiltershape 0 \
  --n_filter_samples 80 \
  --n_fft 256 \
  --formant_supervision 1 \
  --intensity_thres -1 \
  --epoch_num 60 \
  --pretrained_model_dir output/a2a/HB02 \
  --causal 0

In [ ]:
# Check Stage 2 completed
!ls output/e2a/resnet_HB02/*.pth | wc -l
!ls -lh output/e2a/resnet_HB02/model_epoch59.pth

print("\nTRAINING COMPLETE")
print("\nYour pretrained weights:")
print("  output/e2a/resnet_HB02/model_epoch59.pth")

## Next Steps: Use for Phoneme Classification

Update your `ecog_decoder_finetune.ipynb` with:

```python
checkpoint_path = "output/e2a/resnet_HB02/model_epoch59.pth"
```

In [ ]:
loss = 'wandb_export_2025-12-08T14_11_33.921-05_00.csv'

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt

batch_loss = []
with open('wandb_export_2025-12-08T14_11_33.921-05_00.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        batch_loss.append(float(row['e2a_HB02_final - batch_loss']))

batch_loss = np.array(batch_loss)

In [ ]:
plt.plot(np.log(batch_loss))